# 🔢 Linear Algebra — From Fundamentals to ML Applications

Linear algebra is the language of machine learning. This notebook builds from vectors
to eigendecomposition, with every concept connected to a real ML use-case.

**Topics:**
1. Vectors — operations and geometry
2. Matrix operations — multiply, transpose, inverse
3. Systems of linear equations
4. Matrix decompositions — LU, QR, SVD, Eigendecomposition
5. PCA from scratch using SVD
6. Least squares regression
7. PageRank via eigenvectors
8. Low-rank matrix approximation
9. The four fundamental subspaces
10. Norms and distances

In [ ]:
# ── Imports ──────────────────────────────────────────────────────────────────
import numpy as np
import matplotlib.pyplot as plt
from mpl_toolkits.mplot3d import Axes3D   # 3-D visualisation
from numpy.linalg import (norm, det, inv, matrix_rank,
                          solve, eig, svd, lstsq)
import warnings; warnings.filterwarnings('ignore')
np.random.seed(42)
np.set_printoptions(precision=4, suppress=True)
print('Ready.')

## 1. Vectors — Operations and Geometry

In [ ]:
# ── Vector Operations ─────────────────────────────────────────────────────────

# Define two 2-D vectors
a = np.array([3.0, 1.0])   # vector a
b = np.array([1.0, 2.0])   # vector b

# Addition: tip of a + tip of b
add = a + b

# Scalar multiplication: stretch the vector
scaled = 2.5 * a

# L2 norm (Euclidean length): ||a|| = sqrt(a₁² + a₂²)
a_norm = norm(a)

# Dot product: a·b = ||a|| ||b|| cos(θ)
dot = np.dot(a, b)

# Angle between vectors (in degrees)
theta = np.degrees(np.arccos(dot / (norm(a) * norm(b))))

# Cross product (in 3-D, gives a perpendicular vector)
a3 = np.array([3.0, 1.0, 0.0])
b3 = np.array([1.0, 2.0, 0.0])
cross = np.cross(a3, b3)   # perpendicular to both

print(f'a + b       = {add}')
print(f'2.5 * a     = {scaled}')
print(f'||a||       = {a_norm:.4f}')
print(f'a · b       = {dot:.4f}')
print(f'Angle(a,b)  = {theta:.2f}°')
print(f'a × b (z)   = {cross[2]:.4f}  ← area of parallelogram')

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(5, 4))
origin = np.array([0, 0])
for vec, col, label in [(a, '#34d399', 'a'), (b, '#60a5fa', 'b'), (add, '#f59e0b', 'a+b')]:
    ax.annotate('', xy=vec, xytext=origin,
                arrowprops=dict(arrowstyle='->', color=col, lw=2.5))
    ax.text(vec[0]+0.05, vec[1]+0.05, label, color=col, fontsize=12, fontweight='bold')
ax.set_xlim(-0.5, 5); ax.set_ylim(-0.5, 4)
ax.axhline(0, color='gray', lw=0.5); ax.axvline(0, color='gray', lw=0.5)
ax.set_title('Vector addition'); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

## 2. Matrix Operations

In [ ]:
# ── Matrix Operations ─────────────────────────────────────────────────────────

A = np.array([[1, 2, 3],
              [4, 5, 6],
              [7, 8, 10]], dtype=float)  # non-singular

B = np.array([[9, 8, 7],
              [6, 5, 4],
              [3, 2, 1]], dtype=float)

# Transpose: flip rows and columns
print('A.T (transpose):');
print(A.T)

# Matrix–matrix multiplication: (i,k) × (k,j) → (i,j)
C = A @ B
print('\nA @ B:'); print(C)

# Determinant: non-zero ↔ matrix is invertible
print(f'\ndet(A) = {det(A):.4f}')   # non-zero since A is non-singular

# Inverse: A⁻¹ such that A @ A⁻¹ = I
A_inv = inv(A)
print('\nA @ A_inv ≈ I:'); print((A @ A_inv).round(10))  # should be identity

# Rank: number of linearly independent rows/columns
print(f'\nrank(A) = {matrix_rank(A)}')
print(f'rank(B) = {matrix_rank(B)}')  # B has rank < 3 (rows are dependent)

## 3. Solving Linear Systems  Ax = b

In [ ]:
# ── Solving Ax = b ────────────────────────────────────────────────────────────
# Real-world example: 3 chemicals mixed in 3 batches — find concentrations

# A[i,j] = amount of chemical j used in batch i
A_sys = np.array([[2, 1, -1],
                  [-3, -1, 2],
                  [-2, 1,  2]], dtype=float)

b_sys = np.array([8, -11, -3], dtype=float)  # measured outcomes for each batch

# numpy.linalg.solve uses LU decomposition internally
x = solve(A_sys, b_sys)
print(f'Solution x = {x}')            # concentrations of each chemical
print(f'Verify A@x = {A_sys @ x}')    # should equal b_sys
print(f'b_sys      = {b_sys}')

# ── Visualise 2-D case: two lines that intersect at the solution ───────────────
# System: x + 2y = 5 and 3x - y = 1
x_range = np.linspace(-1, 4, 200)
y1 = (5 - x_range) / 2           # from first equation
y2 = 3 * x_range - 1             # from second equation

# Analytical solution
A2 = np.array([[1, 2], [3, -1]])
b2 = np.array([5, 1])
sol = solve(A2, b2)

plt.figure(figsize=(5, 4))
plt.plot(x_range, y1, label='x + 2y = 5', color='#34d399', lw=2)
plt.plot(x_range, y2, label='3x - y = 1', color='#60a5fa', lw=2)
plt.plot(*sol, 'o', color='#f59e0b', ms=10, zorder=5, label=f'Solution ({sol[0]:.2f}, {sol[1]:.2f})')
plt.legend(); plt.ylim(-2, 5); plt.grid(True, alpha=0.3)
plt.title('Ax = b — intersection of two lines')
plt.tight_layout(); plt.show()

## 4. SVD — Singular Value Decomposition

SVD decomposes any matrix M into M = U Σ Vᵀ where:
- **U** : left singular vectors (basis in input space)
- **Σ** : singular values (importance of each component)
- **Vᵀ**: right singular vectors (basis in output space)

Applications: PCA, LSA, recommender systems, image compression.

In [ ]:
# ── Singular Value Decomposition ──────────────────────────────────────────────

# User-movie rating matrix (rows=users, cols=movies)
# 0 means unrated
M = np.array([[5, 4, 0, 1, 0],
              [4, 0, 0, 1, 0],
              [1, 1, 0, 5, 4],
              [0, 0, 5, 4, 4],
              [0, 1, 5, 4, 0]], dtype=float)

# Compute full SVD: M = U @ diag(S) @ Vt
U, S, Vt = svd(M, full_matrices=False)  # economy SVD
print(f'U shape:  {U.shape}')   # (5, 5) left singular vectors
print(f'S shape:  {S.shape}')   # (5,)   singular values (descending)
print(f'Vt shape: {Vt.shape}')  # (5, 5) right singular vectors
print(f'Singular values: {S.round(2)}')

# Fraction of variance explained by each component
var_explained = S**2 / (S**2).sum()
print(f'Variance explained per component: {var_explained.round(3)}')
print(f'Top-2 components explain: {var_explained[:2].sum()*100:.1f}% of variance')

# ── Reconstruct from top-k components (low-rank approximation) ────────────────
def low_rank_approx(U, S, Vt, k):
    """Reconstruct matrix using only the top-k singular components."""
    # Each component is an outer product scaled by its singular value
    return (U[:, :k] * S[:k]) @ Vt[:k, :]

M_k1 = low_rank_approx(U, S, Vt, k=1)
M_k3 = low_rank_approx(U, S, Vt, k=3)

# Frobenius norm of the reconstruction error
err1 = norm(M - M_k1, 'fro')
err3 = norm(M - M_k3, 'fro')
print(f'\nReconstruction error k=1: {err1:.4f}')
print(f'Reconstruction error k=3: {err3:.4f}')

# ── Visualise ─────────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 3, figsize=(12, 3))
for ax, mat, title in [(axes[0], M, 'Original'), (axes[1], M_k1, 'k=1 approx'), (axes[2], M_k3, 'k=3 approx')]:
    im = ax.imshow(mat, cmap='YlOrRd', vmin=0, vmax=5)
    ax.set_title(title); plt.colorbar(im, ax=ax)
plt.suptitle('SVD Low-Rank Approximation of Rating Matrix')
plt.tight_layout(); plt.show()

## 5. PCA from Scratch using SVD

**Principal Component Analysis** finds the directions of maximum variance.
We implement it from scratch using SVD on the centred data matrix.

In [ ]:
# ── PCA from scratch ──────────────────────────────────────────────────────────

# Simulated dataset: 100 samples, 3 correlated features
# Real example: patients with height, weight, BMI — clearly correlated
np.random.seed(7)
height  = np.random.randn(100) * 10 + 170    # cm
weight  = height * 0.5 + np.random.randn(100) * 5   # correlated with height
bmi     = weight / ((height/100)**2) + np.random.randn(100) * 0.5

# Stack into (n_samples, n_features) matrix
X = np.column_stack([height, weight, bmi])  # shape (100, 3)

def pca_from_svd(X, n_components=2):
    """PCA via SVD of the centred data matrix."""
    # Step 1: centre each feature (subtract mean)
    X_centred = X - X.mean(axis=0)        # shape (n, d)

    # Step 2: SVD of centred data (NOT covariance matrix — equivalent but faster)
    U, S, Vt = svd(X_centred, full_matrices=False)

    # Step 3: principal components = columns of V (rows of Vt)
    components = Vt[:n_components]         # shape (n_components, d)

    # Step 4: project data onto principal components
    X_projected = X_centred @ components.T  # shape (n, n_components)

    # Variance explained by each PC
    var_exp = (S**2 / (S**2).sum())[:n_components]
    return X_projected, components, var_exp

X_2d, pcs, var = pca_from_svd(X, n_components=2)
print(f'PC directions (rows):\n{pcs.round(4)}')
print(f'Variance explained: PC1={var[0]*100:.1f}%, PC2={var[1]*100:.1f}%')

plt.figure(figsize=(5, 4))
plt.scatter(X_2d[:, 0], X_2d[:, 1], c=height, cmap='viridis', alpha=0.7, s=30)
plt.colorbar(label='Height (cm)')
plt.xlabel(f'PC1 ({var[0]*100:.0f}% variance)')
plt.ylabel(f'PC2 ({var[1]*100:.0f}% variance)')
plt.title('PCA: height/weight/BMI dataset')
plt.tight_layout(); plt.show()

## 6. Least Squares Regression

Linear regression is a linear algebra problem: find w that minimises ||Xw - y||².

In [ ]:
# ── Least Squares Regression ─────────────────────────────────────────────────
# Example: predict house price from size (m²) and number of rooms

np.random.seed(0)
n = 80
size  = np.random.uniform(50, 200, n)    # m²
rooms = np.random.randint(1, 6, n)       # number of rooms
# True relationship: price = 300*size + 5000*rooms + noise
price = 300 * size + 5000 * rooms + np.random.randn(n) * 8000

# Design matrix X with bias column (intercept)
X_des = np.column_stack([np.ones(n), size, rooms])   # shape (n, 3)

# Closed-form OLS solution: w = (XᵀX)⁻¹ Xᵀy  (via numpy lstsq for stability)
w, residuals, rank, sv = lstsq(X_des, price, rcond=None)
print(f'Intercept:  {w[0]:8.2f}')
print(f'Size coef:  {w[1]:8.2f}  (true: 300)')
print(f'Rooms coef: {w[2]:8.2f}  (true: 5000)')

# Predictions and R²
y_pred = X_des @ w
ss_res = ((price - y_pred)**2).sum()
ss_tot = ((price - price.mean())**2).sum()
r2 = 1 - ss_res / ss_tot
print(f'R² = {r2:.4f}')

plt.figure(figsize=(5, 4))
plt.scatter(y_pred, price, alpha=0.6, color='#60a5fa', s=25)
mn, mx = price.min(), price.max()
plt.plot([mn, mx], [mn, mx], 'r--', lw=1.5, label='Perfect fit')
plt.xlabel('Predicted price'); plt.ylabel('Actual price')
plt.title(f'Least Squares  R²={r2:.3f}')
plt.legend(); plt.tight_layout(); plt.show()

## 7. Eigendecomposition & PageRank

In [ ]:
# ── Eigendecomposition and PageRank ───────────────────────────────────────────
# PageRank assigns importance scores to web pages using the
# principal eigenvector of the link transition matrix.

# 5-node web graph: entry [i,j]=1 means page i links to page j
links = np.array([[0, 1, 1, 0, 0],
                  [0, 0, 1, 1, 0],
                  [1, 0, 0, 1, 0],
                  [0, 0, 0, 0, 1],
                  [1, 1, 0, 0, 0]], dtype=float)

# Transition matrix: normalise rows so each page's link weight sums to 1
row_sums = links.sum(axis=1, keepdims=True)
row_sums[row_sums == 0] = 1  # handle dangling nodes
T = (links / row_sums).T     # transpose: T[j,i] = prob of going from i to j

# Damping factor d = 0.85 (PageRank teleportation — probability of random jump)
d = 0.85
n_pages = 5
# Google matrix: mix T with uniform teleportation matrix
G = d * T + (1 - d) / n_pages * np.ones((n_pages, n_pages))

# Power iteration: multiply by G repeatedly until convergence
rank_vec = np.ones(n_pages) / n_pages   # uniform initialisation
for _ in range(100):
    rank_vec = G @ rank_vec             # each iteration: one matrix-vector product
rank_vec /= rank_vec.sum()              # normalise to sum to 1

print('PageRank scores:')
for i, r in enumerate(rank_vec):
    print(f'  Page {i}: {r:.4f}')

# Verify using eigenvector: should match rank_vec (principal eigenvector of G)
eigenvalues, eigenvectors = eig(G)
# The principal eigenvector corresponds to eigenvalue ≈ 1
idx = np.argmax(eigenvalues.real)
eigen_rank = eigenvectors[:, idx].real
eigen_rank /= eigen_rank.sum()
print('\nEigenvector method (should match):');
print([f'{r:.4f}' for r in eigen_rank])

## 8. Norms and Distances

Different norms are used throughout ML for regularisation, clustering, and similarity search.

In [ ]:
# ── Norms and Distances ───────────────────────────────────────────────────────

v = np.array([3.0, -4.0, 1.0, 2.0])

# L0 norm: count of non-zero elements (sparsity)
l0 = np.count_nonzero(v)
# L1 norm: sum of absolute values — promotes sparsity (used in Lasso)
l1 = norm(v, ord=1)
# L2 norm: Euclidean length — most common (used in Ridge, cosine similarity)
l2 = norm(v, ord=2)
# L∞ norm: maximum absolute value
linf = norm(v, ord=np.inf)

print(f'Vector: {v}')
print(f'L0 norm (sparsity):   {l0}')
print(f'L1 norm (Manhattan):  {l1:.4f}')
print(f'L2 norm (Euclidean):  {l2:.4f}')
print(f'L∞ norm (max-abs):    {linf:.4f}')

# ── Frobenius norm for matrices ───────────────────────────────────────────────
W = np.random.randn(4, 4)  # weight matrix
frob = norm(W, 'fro')      # sqrt of sum of squared entries — used in L2 regularisation
print(f'\nFrobenius norm of 4×4 weight matrix: {frob:.4f}')

# ── Distance metrics: L1 vs L2 ────────────────────────────────────────────────
# Grid of points on the unit circle for each norm
theta = np.linspace(0, 2*np.pi, 500)
# L1 unit ball: |x| + |y| = 1
# L2 unit ball: x² + y² = 1
x_l2 = np.cos(theta); y_l2 = np.sin(theta)

# L1 unit ball — diamond shape
x_l1 = np.array([1, 0, -1, 0, 1])
y_l1 = np.array([0, 1, 0, -1, 0])

plt.figure(figsize=(5, 5))
plt.plot(x_l2, y_l2, label='L2 ball (circle)', color='#34d399', lw=2)
plt.plot(x_l1, y_l1, label='L1 ball (diamond)', color='#60a5fa', lw=2)
plt.axhline(0, color='gray', lw=0.5); plt.axvline(0, color='gray', lw=0.5)
plt.legend(); plt.title('L1 vs L2 unit balls'); plt.axis('equal')
plt.tight_layout(); plt.show()

## Summary

| Concept | Formula | ML Application |
|---|---|---|
| Dot product | a·b = Σaᵢbᵢ | Attention, similarity |
| Cosine similarity | a·b / (‖a‖‖b‖) | Semantic search |
| SVD | M = UΣVᵀ | PCA, recommender systems |
| Low-rank approx | UₖΣₖVₖᵀ | Image/text compression |
| Least squares | w = (XᵀX)⁻¹Xᵀy | Linear regression |
| Eigendecomposition | Av = λv | PageRank, PCA |
| L1 norm | Σ|wᵢ| | Lasso sparsity |
| L2 norm | √(Σwᵢ²) | Ridge, weight decay |